# R2-08 · Pipelines reproducibles — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: el mismo pipeline sobre otro dato del Estado — la **matrícula
escolar 2025** (MINEDUC). Dos variables **categóricas** con one-hot, una línea base que superar,
y un pipeline guardado que **no se cae** ante una categoría nunca vista.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, numpy as np, os, urllib.request, joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

CSV = "matricula.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/matricula.csv", CSV)
mat = pd.read_csv(CSV)

# Tarea: estimar la matrícula total de un establecimiento a partir de sus cursos,
# si es rural, su dependencia y su región.
NUM = ["cursos", "rural"]
CAT = ["dependencia", "region"]
X_tr, X_te, y_tr, y_te = train_test_split(mat[NUM + CAT], mat["mat_total"],
                                          test_size=0.3, random_state=42)
print("train:", X_tr.shape, "| test:", X_te.shape)
print(mat["dependencia"].value_counts())

## 1. Transferencia: un preprocesador con dos variables categóricas

Dominio nuevo: la **matrícula escolar 2025** (MINEDUC). Ahora hay **dos** categóricas (`dependencia`, con 5 valores, y `region`, con 16) y dos numéricas. Misma receta: escalar las numéricas y one-hot a las categóricas.

### ✍️ Ejercicio 1 — Arma el ColumnTransformer

In [ ]:
pre = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])
n_columnas = pre.fit_transform(X_tr).shape[1]
print("columnas tras preprocesar:", n_columnas)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert n_columnas == len(NUM) + X_tr["dependencia"].nunique() + X_tr["region"].nunique()
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "('num', StandardScaler(), NUM) y ('cat', OneHotEncoder(handle_unknown='ignore'), CAT).")
    print("   Detalle:", e)

## 2. Pipeline y línea base

Encadena el preprocesador con un bosque y compáralo con la línea base más simple: predecir para todos la **mediana** de matrícula del entrenamiento.

### ✍️ Ejercicio 2 — Entrena el pipeline y supera a la mediana

In [ ]:
pipe = Pipeline([("pre", pre),
                 ("modelo", RandomForestRegressor(n_estimators=50, min_samples_leaf=5,
                                                  random_state=42, n_jobs=-1))]).fit(X_tr, y_tr)
mae_pipe = mean_absolute_error(y_te, pipe.predict(X_te))
mae_base = mean_absolute_error(y_te, np.full(len(y_te), y_tr.median()))
print(f"MAE pipeline: {mae_pipe:.0f} estudiantes | MAE mediana: {mae_base:.0f} estudiantes")

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert [n for n, _ in pipe.steps] == ["pre", "modelo"]
    assert mae_pipe < 0.6 * mae_base, "El pipeline debería bajar mucho el error de la mediana"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Pipeline([...]).fit(X_tr, y_tr); mean_absolute_error(y_te, pipe.predict(X_te)); np.full(len(y_te), y_tr.median()) para la línea base.')
    print("   Detalle:", e)

## 3. Guardar, recargar y resistir una categoría nunca vista

Guarda el pipeline como `modelo_matricula.joblib`, recárgalo y predice dos establecimientos nuevos. El segundo trae una región mal escrita («Region Inventada»): gracias a `handle_unknown="ignore"` el pipeline **no se cae** (la trata como «ninguna región conocida»). Ojo: no fallar no es lo mismo que acertar — en producción conviene **validar** las entradas (R2-09).

### ✍️ Ejercicio 3 — Persistencia y robustez ante datos nuevos

In [ ]:
joblib.dump(pipe, "modelo_matricula.joblib")
pipe2 = joblib.load("modelo_matricula.joblib")
nuevos = pd.DataFrame({"cursos": [12, 4], "rural": [0, 1],
                       "dependencia": ["Servicio Local de Educación", "Municipal"],
                       "region": ["Metropolitana", "Region Inventada"]})
pred = pipe2.predict(nuevos)
print(pred.round(0))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert os.path.exists("modelo_matricula.joblib")
    assert len(pred) == 2 and (np.asarray(pred) > 0).all()
    assert np.allclose(pred, pipe.predict(nuevos))
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "joblib.dump(pipe, 'modelo_matricula.joblib'); pipe2 = joblib.load('modelo_matricula.joblib'); pred = pipe2.predict(nuevos).")
    print("   Detalle:", e)

---

*Fin de la profundización de R2-08.*

*Fuente: MINEDUC, Resumen de matrícula por establecimiento 2025 (datos abiertos).*